# Polymer Analysis with ChemiAccel

This notebook demonstrates ChemiAccel's specialized capabilities for polymer science, including monomer analysis, copolymer characterization, property prediction, and inverse design.

## Prerequisites

```bash
pip install chemi-accel[cuda]  # For GPU acceleration
```

In [ ]:
# Import required libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from chemi_accel.polymer.copolymer import CopolymerAnalyzer
from chemi_accel.polymer.inverse_design import PolymerInverseDesign
from chemi_accel.polymer.monomer import MonomerAnalyzer
from chemi_accel.polymer.predictive.property_engine import PropertyEngine

# Set up plotting
plt.style.use("seaborn-v0_8")
sns.set_palette("husl")

print("ChemiAccel Polymer Analysis Tutorial")
print("====================================")

## 1. Monomer Reactivity Analysis

Let's analyze the reactivity and polymerization characteristics of common monomers.

In [ ]:
# Initialize monomer analyzer
monomer_analyzer = MonomerAnalyzer()

# Define common monomers
monomers = {
    "Styrene": "C=Cc1ccccc1",
    "Methyl Methacrylate": "C=C(C)C(=O)OC",
    "Acrylonitrile": "C=CC#N",
    "Vinyl Acetate": "C=COC(=O)C",
    "Ethylene": "C=C",
    "Propylene": "C=CC",
    "Butadiene": "C=CC=C",
    "Isoprene": "C=C(C)C=C",
}

print("=== MONOMER REACTIVITY ANALYSIS ===")

monomer_data = []
for name, smiles in monomers.items():
    analysis = monomer_analyzer.analyze_monomer(smiles)

    data = {
        "name": name,
        "smiles": smiles,
        "mechanism": analysis["mechanism"],
        "reactivity_ratio": analysis["reactivity_ratio"],
        "ceiling_temp": analysis["ceiling_temp"],
        "delta_h_poly": analysis["delta_h_poly"],
    }
    monomer_data.append(data)

    print(f"\n{name}:")
    print(f"  SMILES: {smiles}")
    print(f"  Polymerization mechanism: {analysis['mechanism']}")
    print(f"  Reactivity ratio: {analysis['reactivity_ratio']:.3f}")
    print(f"  Ceiling temperature: {analysis['ceiling_temp']:.1f} K")
    print(f"  ΔH polymerization: {analysis['delta_h_poly']:.1f} kJ/mol")

monomer_df = pd.DataFrame(monomer_data)
display(monomer_df)

## 2. Visualizing Monomer Properties

In [ ]:
# Create comprehensive monomer analysis plots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Monomer Reactivity Analysis", fontsize=16, fontweight="bold")

# 1. Reactivity ratios
bars1 = axes[0, 0].barh(
    monomer_df["name"], monomer_df["reactivity_ratio"], color="lightblue", alpha=0.7
)
axes[0, 0].set_xlabel("Reactivity Ratio")
axes[0, 0].set_title("Monomer Reactivity Ratios")
axes[0, 0].grid(True, alpha=0.3)

# Add value labels
for i, (bar, value) in enumerate(zip(bars1, monomer_df["reactivity_ratio"])):
    axes[0, 0].text(
        bar.get_width() + 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{value:.3f}",
        va="center",
        fontsize=9,
    )

# 2. Ceiling temperatures
bars2 = axes[0, 1].bar(
    range(len(monomer_df)), monomer_df["ceiling_temp"], color="lightcoral", alpha=0.7
)
axes[0, 1].set_xlabel("Monomer")
axes[0, 1].set_ylabel("Ceiling Temperature (K)")
axes[0, 1].set_title("Ceiling Temperatures")
axes[0, 1].set_xticks(range(len(monomer_df)))
axes[0, 1].set_xticklabels(
    [name[:8] for name in monomer_df["name"]], rotation=45, ha="right"
)

# 3. Polymerization enthalpy
bars3 = axes[1, 0].bar(
    range(len(monomer_df)), monomer_df["delta_h_poly"], color="lightgreen", alpha=0.7
)
axes[1, 0].set_xlabel("Monomer")
axes[1, 0].set_ylabel("ΔH Polymerization (kJ/mol)")
axes[1, 0].set_title("Polymerization Enthalpy")
axes[1, 0].set_xticks(range(len(monomer_df)))
axes[1, 0].set_xticklabels(
    [name[:8] for name in monomer_df["name"]], rotation=45, ha="right"
)

# 4. Mechanism pie chart
mechanism_counts = monomer_df["mechanism"].value_counts()
axes[1, 1].pie(
    mechanism_counts.values,
    labels=mechanism_counts.index,
    autopct="%1.1f%%",
    colors=["gold", "lightblue", "lightcoral", "lightgreen"],
)
axes[1, 1].set_title("Polymerization Mechanisms")

plt.tight_layout()
plt.show()

print("\nMonomer Analysis Summary:")
print(
    f"Most reactive monomer: {monomer_df.loc[monomer_df['reactivity_ratio'].idxmax(), 'name']}"
)
print(
    f"Highest ceiling temperature: {monomer_df.loc[monomer_df['ceiling_temp'].idxmax(), 'name']}"
)
print(
    f"Most exothermic polymerization: {monomer_df.loc[monomer_df['delta_h_poly'].idxmin(), 'name']}"
)

## 3. Copolymer Composition Analysis

Let's analyze the composition drift and sequence distribution in copolymer systems.

In [ ]:
# Initialize copolymer analyzer
copolymer_analyzer = CopolymerAnalyzer()

print("=== COPOLYMER COMPOSITION ANALYSIS ===")

# Define copolymer system: Styrene-Acrylonitrile (SAN)
monomer_1 = "C=Cc1ccccc1"  # Styrene
monomer_2 = "C=CC#N"  # Acrylonitrile
feed_ratio_1 = 0.7  # 70% styrene feed
feed_ratio_2 = 0.3  # 30% acrylonitrile feed

print(f"Copolymer System: Styrene-Acrylonitrile")
print(f"Feed composition: {feed_ratio_1:.1%} styrene, {feed_ratio_2:.1%} acrylonitrile")

# Analyze composition drift
composition_data = copolymer_analyzer.analyze_composition_drift(
    monomer_1=monomer_1,
    monomer_2=monomer_2,
    feed_ratio_1=feed_ratio_1,
    feed_ratio_2=feed_ratio_2,
    conversion_range=(0, 0.95),
    points=50,
)

# Extract data for plotting
conversions = [point["conversion"] for point in composition_data["composition_curve"]]
inst_f1 = [point["instantaneous_f1"] for point in composition_data["composition_curve"]]
cum_F1 = [point["cumulative_F1"] for point in composition_data["composition_curve"]]

# Sequence distribution analysis
sequence_analysis = copolymer_analyzer.analyze_sequence_distribution(
    monomer_1=monomer_1,
    monomer_2=monomer_2,
    feed_ratio_1=feed_ratio_1,
    feed_ratio_2=feed_ratio_2,
)

print(f"\nSequence Distribution Analysis:")
print(f"  Average sequence length (styrene): {sequence_analysis['ln1']:.2f}")
print(f"  Average sequence length (acrylonitrile): {sequence_analysis['ln2']:.2f}")
print(f"  Probability of alternation: {sequence_analysis['prob_alternation']:.3f}")
print(f"  Randomness parameter: {sequence_analysis['randomness']:.3f}")

# Plot composition drift
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Composition vs conversion
ax1.plot(
    conversions,
    inst_f1,
    "b-",
    linewidth=2,
    label="Instantaneous (f₁)",
    marker="o",
    markersize=4,
)
ax1.plot(
    conversions,
    cum_F1,
    "r--",
    linewidth=2,
    label="Cumulative (F₁)",
    marker="s",
    markersize=4,
)
ax1.axhline(
    y=feed_ratio_1, color="gray", linestyle=":", alpha=0.7, label="Feed composition"
)
ax1.set_xlabel("Conversion")
ax1.set_ylabel("Mole Fraction of Styrene")
ax1.set_title("Composition Drift in SAN Copolymer")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 1)
ax1.set_ylim(0, 1)

# Sequence distribution visualization
sequence_lengths = np.arange(1, 11)
styrene_sequences = [
    sequence_analysis["ln1"] * np.exp(-i / sequence_analysis["ln1"])
    for i in sequence_lengths
]
an_sequences = [
    sequence_analysis["ln2"] * np.exp(-i / sequence_analysis["ln2"])
    for i in sequence_lengths
]

x = np.arange(len(sequence_lengths))
width = 0.35

ax2.bar(
    x - width / 2,
    styrene_sequences,
    width,
    label="Styrene sequences",
    alpha=0.7,
    color="blue",
)
ax2.bar(
    x + width / 2,
    an_sequences,
    width,
    label="Acrylonitrile sequences",
    alpha=0.7,
    color="red",
)
ax2.set_xlabel("Sequence Length")
ax2.set_ylabel("Probability Density")
ax2.set_title("Sequence Length Distribution")
ax2.set_xticks(x)
ax2.set_xticklabels(sequence_lengths)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Polymer Property Prediction

Let's predict the properties of various polymer structures.

In [ ]:
# Initialize property prediction engine
property_engine = PropertyEngine()

print("=== POLYMER PROPERTY PREDICTION ===")

# Define common polymer structures
polymers = {
    "Polystyrene": "[CH2CH(c1ccccc1)]n",
    "PMMA": "[CH2C(C)(C(=O)OC)]n",
    "Polyethylene": "[CH2CH2]n",
    "Polypropylene": "[CH2CH(C)]n",
    "PVC": "[CH2CHCl]n",
    "PAN": "[CH2CH(C#N)]n",
    "PTFE": "[CF2CF2]n",
    "Nylon-6": "[NH(CH2)5C(=O)]n",
}

polymer_properties = []
for name, structure in polymers.items():
    properties = property_engine.predict_properties(structure)

    data = {
        "name": name,
        "structure": structure,
        "tg_k": properties["tg"],
        "tg_c": properties["tg"] - 273.15,
        "density": properties["density"],
        "elastic_modulus": properties["elastic_modulus"],
        "thermal_expansion": properties["thermal_expansion"],
        "solubility_parameter": properties.get("solubility_parameter", np.nan),
    }
    polymer_properties.append(data)

    print(f"\n{name}:")
    print(
        f"  Glass transition temperature: {properties['tg']:.1f} K ({properties['tg']-273.15:.1f} °C)"
    )
    print(f"  Density: {properties['density']:.3f} g/cm³")
    print(f"  Elastic modulus: {properties['elastic_modulus']:.0f} MPa")
    print(f"  Thermal expansion: {properties['thermal_expansion']:.1e} K⁻¹")

polymer_df = pd.DataFrame(polymer_properties)
display(polymer_df[["name", "tg_c", "density", "elastic_modulus"]])

## 5. Polymer Property Visualization and Analysis

In [ ]:
# Create comprehensive polymer property analysis
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle("Polymer Property Analysis", fontsize=16, fontweight="bold")

# 1. Glass transition temperatures
colors = plt.cm.viridis(np.linspace(0, 1, len(polymer_df)))
bars1 = axes[0, 0].barh(polymer_df["name"], polymer_df["tg_c"], color=colors, alpha=0.7)
axes[0, 0].set_xlabel("Glass Transition Temperature (°C)")
axes[0, 0].set_title("Glass Transition Temperatures")
axes[0, 0].grid(True, alpha=0.3)

# Add temperature labels
for bar, temp in zip(bars1, polymer_df["tg_c"]):
    axes[0, 0].text(
        bar.get_width() + 5,
        bar.get_y() + bar.get_height() / 2,
        f"{temp:.0f}°C",
        va="center",
        fontsize=9,
    )

# 2. Density vs Elastic Modulus
scatter = axes[0, 1].scatter(
    polymer_df["density"],
    polymer_df["elastic_modulus"],
    s=100,
    c=polymer_df["tg_c"],
    cmap="coolwarm",
    alpha=0.7,
)
axes[0, 1].set_xlabel("Density (g/cm³)")
axes[0, 1].set_ylabel("Elastic Modulus (MPa)")
axes[0, 1].set_title("Density vs Elastic Modulus\n(Color: Tg)")
cbar = plt.colorbar(scatter, ax=axes[0, 1])
cbar.set_label("Tg (°C)")

# Add polymer labels
for i, name in enumerate(polymer_df["name"]):
    axes[0, 1].annotate(
        name[:4],
        (polymer_df["density"].iloc[i], polymer_df["elastic_modulus"].iloc[i]),
        xytext=(5, 5),
        textcoords="offset points",
        fontsize=8,
    )

# 3. Property comparison radar chart for selected polymers
from math import pi

# Select 4 representative polymers
selected_polymers = ["Polystyrene", "PMMA", "Polyethylene", "PTFE"]
selected_data = polymer_df[polymer_df["name"].isin(selected_polymers)].copy()

# Normalize properties for radar chart
properties_radar = ["tg_c", "density", "elastic_modulus"]
norm_data = selected_data[properties_radar].copy()

for prop in properties_radar:
    min_val = norm_data[prop].min()
    max_val = norm_data[prop].max()
    if max_val > min_val:
        norm_data[prop] = (norm_data[prop] - min_val) / (max_val - min_val)
    else:
        norm_data[prop] = 0.5

# Create radar chart on subplot
ax_radar = plt.subplot(2, 2, 3, projection="polar")
N = len(properties_radar)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]

colors_radar = ["red", "blue", "green", "orange"]
for i, (idx, polymer) in enumerate(selected_data.iterrows()):
    values = norm_data.iloc[i].tolist()
    values += values[:1]

    ax_radar.plot(
        angles,
        values,
        "o-",
        linewidth=2,
        label=polymer["name"][:6],
        color=colors_radar[i],
    )
    ax_radar.fill(angles, values, alpha=0.25, color=colors_radar[i])

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(["Tg", "Density", "E-modulus"])
ax_radar.set_ylim(0, 1)
ax_radar.set_title("Property Comparison", pad=20)
ax_radar.legend(loc="upper right", bbox_to_anchor=(1.2, 1.0))

# 4. Thermal expansion coefficients
thermal_exp_data = polymer_df.dropna(subset=["thermal_expansion"])
bars4 = axes[1, 1].bar(
    range(len(thermal_exp_data)),
    thermal_exp_data["thermal_expansion"],
    color="orange",
    alpha=0.7,
)
axes[1, 1].set_xlabel("Polymer")
axes[1, 1].set_ylabel("Thermal Expansion (K⁻¹)")
axes[1, 1].set_title("Thermal Expansion Coefficients")
axes[1, 1].set_xticks(range(len(thermal_exp_data)))
axes[1, 1].set_xticklabels(
    [name[:6] for name in thermal_exp_data["name"]], rotation=45, ha="right"
)
axes[1, 1].ticklabel_format(style="scientific", axis="y", scilimits=(0, 0))

plt.tight_layout()
plt.show()

# Property analysis summary
print("\nPolymer Property Summary:")
print(
    f"Highest Tg: {polymer_df.loc[polymer_df['tg_c'].idxmax(), 'name']} ({polymer_df['tg_c'].max():.0f}°C)"
)
print(
    f"Lowest Tg: {polymer_df.loc[polymer_df['tg_c'].idxmin(), 'name']} ({polymer_df['tg_c'].min():.0f}°C)"
)
print(
    f"Highest density: {polymer_df.loc[polymer_df['density'].idxmax(), 'name']} ({polymer_df['density'].max():.3f} g/cm³)"
)
print(
    f"Stiffest: {polymer_df.loc[polymer_df['elastic_modulus'].idxmax(), 'name']} ({polymer_df['elastic_modulus'].max():.0f} MPa)"
)

## 6. Polymer Inverse Design

Let's design polymers with specific target properties using AI-driven inverse design.

In [ ]:
# Initialize polymer inverse design
polymer_designer = PolymerInverseDesign(
    algorithm="genetic_algorithm",
    population_size=50,  # Smaller for demo
    generations=20,  # Fewer generations for demo
)

print("=== POLYMER INVERSE DESIGN ===")

# Define target properties for a high-performance polymer
target_properties = {
    "glass_transition_temp": 150 + 273.15,  # 150°C in Kelvin
    "density": 1.3,  # g/cm³
    "elastic_modulus": 3000,  # MPa
    "thermal_conductivity": 0.5,  # W/m·K
    "dielectric_constant": 3.0,  # -
}

# Property weights (importance)
property_weights = {
    "glass_transition_temp": 3.0,  # Very important
    "density": 1.0,
    "elastic_modulus": 2.5,  # Very important
    "thermal_conductivity": 2.0,
    "dielectric_constant": 1.5,
}

print("Target Properties:")
for prop, value in target_properties.items():
    if prop == "glass_transition_temp":
        print(f"  {prop}: {value:.1f} K ({value-273.15:.0f}°C)")
    else:
        print(f"  {prop}: {value}")

print("\nRunning inverse design optimization...")

# Generate polymer candidates
candidates = polymer_designer.generate_polymers(
    target_properties=target_properties,
    property_weights=property_weights,
    constraint_functions={
        "synthesizability": lambda x: x < 4.0,  # Reasonably synthesizable
        "stability": lambda x: x > 0.7,  # Stable
    },
)

print(f"\nGenerated {len(candidates)} polymer candidates")
print("\nTop 5 Candidates:")

candidate_data = []
for i, candidate in enumerate(candidates[:5]):
    props = candidate["predicted_properties"]

    data = {
        "rank": i + 1,
        "structure": candidate["structure"],
        "fitness": candidate["fitness"],
        "tg_c": props["glass_transition_temp"] - 273.15,
        "density": props["density"],
        "elastic_modulus": props["elastic_modulus"],
        "thermal_conductivity": props["thermal_conductivity"],
        "dielectric_constant": props["dielectric_constant"],
    }
    candidate_data.append(data)

    print(f"\n{i+1}. {candidate['structure']}")
    print(f"   Fitness score: {candidate['fitness']:.3f}")
    print(
        f"   Predicted Tg: {props['glass_transition_temp']-273.15:.1f}°C (target: {target_properties['glass_transition_temp']-273.15:.0f}°C)"
    )
    print(
        f"   Predicted density: {props['density']:.3f} g/cm³ (target: {target_properties['density']:.1f})"
    )
    print(
        f"   Predicted E-modulus: {props['elastic_modulus']:.0f} MPa (target: {target_properties['elastic_modulus']:.0f})"
    )

candidate_df = pd.DataFrame(candidate_data)
display(
    candidate_df[["rank", "structure", "fitness", "tg_c", "density", "elastic_modulus"]]
)

## 7. Design Performance Analysis

In [ ]:
# Analyze how well the designed polymers meet the targets
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Inverse Design Performance Analysis", fontsize=16, fontweight="bold")

targets = {
    "tg_c": target_properties["glass_transition_temp"] - 273.15,
    "density": target_properties["density"],
    "elastic_modulus": target_properties["elastic_modulus"],
    "thermal_conductivity": target_properties["thermal_conductivity"],
}

# 1. Tg comparison
x_pos = range(len(candidate_df))
bars1 = axes[0, 0].bar(
    x_pos, candidate_df["tg_c"], alpha=0.7, color="skyblue", label="Predicted"
)
axes[0, 0].axhline(
    y=targets["tg_c"], color="red", linestyle="--", linewidth=2, label="Target"
)
axes[0, 0].set_xlabel("Candidate Rank")
axes[0, 0].set_ylabel("Glass Transition Temperature (°C)")
axes[0, 0].set_title("Glass Transition Temperature")
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# 2. Density comparison
bars2 = axes[0, 1].bar(
    x_pos, candidate_df["density"], alpha=0.7, color="lightcoral", label="Predicted"
)
axes[0, 1].axhline(
    y=targets["density"], color="red", linestyle="--", linewidth=2, label="Target"
)
axes[0, 1].set_xlabel("Candidate Rank")
axes[0, 1].set_ylabel("Density (g/cm³)")
axes[0, 1].set_title("Density")
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# 3. Elastic modulus comparison
bars3 = axes[1, 0].bar(
    x_pos,
    candidate_df["elastic_modulus"],
    alpha=0.7,
    color="lightgreen",
    label="Predicted",
)
axes[1, 0].axhline(
    y=targets["elastic_modulus"],
    color="red",
    linestyle="--",
    linewidth=2,
    label="Target",
)
axes[1, 0].set_xlabel("Candidate Rank")
axes[1, 0].set_ylabel("Elastic Modulus (MPa)")
axes[1, 0].set_title("Elastic Modulus")
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# 4. Overall fitness scores
bars4 = axes[1, 1].bar(x_pos, candidate_df["fitness"], alpha=0.7, color="gold")
axes[1, 1].set_xlabel("Candidate Rank")
axes[1, 1].set_ylabel("Fitness Score")
axes[1, 1].set_title("Overall Fitness Score")
axes[1, 1].grid(True, alpha=0.3)

# Add fitness values on bars
for bar, fitness in zip(bars4, candidate_df["fitness"]):
    axes[1, 1].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{fitness:.3f}",
        ha="center",
        va="bottom",
        fontsize=9,
    )

plt.tight_layout()
plt.show()

# Calculate prediction errors
print("\nPrediction Accuracy Analysis:")
for i, candidate in enumerate(candidate_df.itertuples()):
    tg_error = abs(candidate.tg_c - targets["tg_c"]) / targets["tg_c"] * 100
    density_error = (
        abs(candidate.density - targets["density"]) / targets["density"] * 100
    )
    modulus_error = (
        abs(candidate.elastic_modulus - targets["elastic_modulus"])
        / targets["elastic_modulus"]
        * 100
    )

    print(f"\nCandidate {candidate.rank}:")
    print(f"  Tg error: {tg_error:.1f}%")
    print(f"  Density error: {density_error:.1f}%")
    print(f"  E-modulus error: {modulus_error:.1f}%")
    print(f"  Average error: {(tg_error + density_error + modulus_error)/3:.1f}%")

## 8. Interactive Polymer Designer

Create an interactive tool for exploring different target properties.

In [ ]:
from IPython.widgets import Dropdown, FloatSlider, interact


def interactive_polymer_design(
    target_tg=150, target_density=1.3, target_modulus=3000, application_type="General"
):
    """Interactive polymer design function"""

    print(f"Designing polymer for: {application_type}")
    print(f"Target Tg: {target_tg}°C")
    print(f"Target Density: {target_density} g/cm³")
    print(f"Target Elastic Modulus: {target_modulus} MPa")
    print("=" * 50)

    # Adjust targets based on application
    if application_type == "High Temperature":
        target_tg = max(target_tg, 200)
        target_modulus = max(target_modulus, 4000)
    elif application_type == "Flexible":
        target_tg = min(target_tg, 50)
        target_modulus = min(target_modulus, 1000)
    elif application_type == "Lightweight":
        target_density = min(target_density, 1.0)

    # Quick design simulation (simplified for demo)
    design_targets = {
        "glass_transition_temp": target_tg + 273.15,
        "density": target_density,
        "elastic_modulus": target_modulus,
    }

    # Simulate design results
    np.random.seed(42)  # For reproducible results
    simulated_results = {
        "tg": target_tg + np.random.normal(0, 10),
        "density": target_density + np.random.normal(0, 0.1),
        "modulus": target_modulus + np.random.normal(0, 200),
    }

    print(f"Predicted Properties:")
    print(f"  Tg: {simulated_results['tg']:.1f}°C (target: {target_tg}°C)")
    print(
        f"  Density: {simulated_results['density']:.3f} g/cm³ (target: {target_density})"
    )
    print(
        f"  E-modulus: {simulated_results['modulus']:.0f} MPa (target: {target_modulus})"
    )

    # Calculate fitness
    tg_error = abs(simulated_results["tg"] - target_tg) / target_tg
    density_error = abs(simulated_results["density"] - target_density) / target_density
    modulus_error = abs(simulated_results["modulus"] - target_modulus) / target_modulus
    overall_fitness = 1 / (1 + tg_error + density_error + modulus_error)

    print(f"\nDesign Fitness: {overall_fitness:.3f}")

    if overall_fitness > 0.8:
        print("✅ Excellent design match!")
    elif overall_fitness > 0.6:
        print("✓ Good design match")
    else:
        print("⚠ Design needs optimization")

    # Simple visualization
    fig, ax = plt.subplots(figsize=(8, 4))

    properties = ["Tg (°C)", "Density\n(g/cm³)", "E-modulus\n(MPa/1000)"]
    targets = [target_tg, target_density, target_modulus / 1000]
    predicted = [
        simulated_results["tg"],
        simulated_results["density"],
        simulated_results["modulus"] / 1000,
    ]

    x = np.arange(len(properties))
    width = 0.35

    ax.bar(x - width / 2, targets, width, label="Target", alpha=0.7, color="lightblue")
    ax.bar(
        x + width / 2,
        predicted,
        width,
        label="Predicted",
        alpha=0.7,
        color="lightcoral",
    )

    ax.set_xlabel("Properties")
    ax.set_ylabel("Values")
    ax.set_title(f"Design Results for {application_type} Application")
    ax.set_xticks(x)
    ax.set_xticklabels(properties)
    ax.legend()
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Create interactive widget
print("Interactive Polymer Designer")
print("Adjust the sliders to explore different target properties:")

interact(
    interactive_polymer_design,
    target_tg=FloatSlider(
        min=-50, max=300, step=10, value=150, description="Target Tg (°C):"
    ),
    target_density=FloatSlider(
        min=0.5, max=2.0, step=0.1, value=1.3, description="Density (g/cm³):"
    ),
    target_modulus=FloatSlider(
        min=100, max=10000, step=100, value=3000, description="E-modulus (MPa):"
    ),
    application_type=Dropdown(
        options=["General", "High Temperature", "Flexible", "Lightweight"],
        value="General",
        description="Application:",
    ),
);

## 9. Summary and Next Steps

In this notebook, we've explored ChemiAccel's comprehensive polymer analysis capabilities:

### What We Covered
- **Monomer Analysis**: Reactivity ratios and polymerization mechanisms
- **Copolymer Characterization**: Composition drift and sequence distribution
- **Property Prediction**: Glass transition temperatures, mechanical properties
- **Inverse Design**: AI-driven polymer design for target properties
- **Interactive Tools**: Real-time polymer design exploration

### Advanced Applications
- **Thermoset Analysis**: Crosslink density and network properties
- **Conformational Analysis**: Chain flexibility and dynamics
- **High-Throughput Screening**: Large-scale polymer libraries
- **Multi-Objective Optimization**: Balancing competing properties

### Next Steps
- Explore the [Polymer Science Guide](../domains/polymer_science.rst)
- Try the [Advanced Polymer Examples](../examples/polymer_property_prediction.py)
- Learn about [Materials Science](materials_discovery.ipynb)
- Discover [AI Workflows](ai_workflows.ipynb)

In [ ]:
print("🎉 Polymer Analysis Tutorial Complete!")
print("\nYou've learned how to:")
print("✅ Analyze monomer reactivity and polymerization")
print("✅ Characterize copolymer systems")
print("✅ Predict polymer properties")
print("✅ Design polymers with target properties")
print("✅ Use interactive analysis tools")
print("\nReady to design the next generation of polymers! 🧪")